%md
## Step 1: Fix column types
Bronze stored everything as text.
Here I change the IDs and time steps to whole numbers, and the measurement columns to decimal numbers.

In [0]:
from pyspark.sql.functions import col

# Load the bronze table
df = spark.table("bronze.elliptic_txs_features")

# Change ID and time step to whole numbers
df = df.withColumn("txId", col("txId").cast("bigint"))
df = df.withColumn("time_step", col("time_step").cast("int"))

# Go through every column and change the measurement columns to decimal numbers
for name in df.columns:
    if name.startswith("local_") or name.startswith("agg_"):
        df = df.withColumn(name, col(name).cast("double"))

# Save as a silver table and count the rows
df.write.mode("overwrite").saveAsTable("silver.elliptic_txs_features")
print(df.count())

%md
## Step 2: Clean the labels table
Change the ID to a whole number.
Give each class a clear name: 1 means illicit, 2 means licit, and unknown stays unknown.

In [0]:
from pyspark.sql.functions import col, when

# Load the labels table
labels = spark.table("bronze.elliptic_txs_classes")

# Change ID to a whole number
labels = labels.withColumn("txId", col("txId").cast("bigint"))

# Give each class a clear name
labels = labels.withColumn(
    "label_name",
    when(col("class") == "1", "illicit")
    .when(col("class") == "2", "licit")
    .otherwise("unknown")
)

# Save as a silver table and count the rows
labels.write.mode("overwrite").saveAsTable("silver.elliptic_txs_classes")
print(labels.count())

%md
## Step 3: Clean the edges table
The edges table lists which transactions connect to which.
Change both IDs to whole numbers, then check that every ID exists in the features table.

In [0]:
from pyspark.sql.functions import col

edges = spark.table("bronze.elliptic_txs_edgelist")

# Change both IDs to whole numbers
edges = edges.withColumn("txId1", col("txId1").cast("bigint"))
edges = edges.withColumn("txId2", col("txId2").cast("bigint"))

# Save as a silver table and count the rows
edges.write.mode("overwrite").saveAsTable("silver.elliptic_txs_edgelist")
print(edges.count())

In [0]:
%sql
SELECT COUNT(*) AS broken_links
FROM silver.elliptic_txs_edgelist
WHERE txId1 NOT IN (SELECT txId FROM silver.elliptic_txs_features)
   OR txId2 NOT IN (SELECT txId FROM silver.elliptic_txs_features)

%md
## Step 4: Quality checks
We count a few things to make sure the silver tables are correct.
Expected results: 203769 rows, 203769 unique IDs, 0 missing IDs, 49 time steps, and 0 transactions without a label.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM silver.elliptic_txs_features) AS feature_rows,
  (SELECT COUNT(DISTINCT txId) FROM silver.elliptic_txs_features) AS unique_ids,
  (SELECT COUNT(*) FROM silver.elliptic_txs_features WHERE txId IS NULL OR time_step IS NULL) AS missing_ids,
  (SELECT COUNT(DISTINCT time_step) FROM silver.elliptic_txs_features) AS time_steps,
  (SELECT COUNT(*)
     FROM silver.elliptic_txs_features f
     LEFT JOIN silver.elliptic_txs_classes c ON f.txId = c.txId
     WHERE c.txId IS NULL) AS no_label

%md
## Step 5: Join labels to features
Each transaction gets its label (illicit, licit, or unknown) added next to its features.
We keep one row per transaction. Expected result: 203769 rows.


In [0]:
features = spark.table("silver.elliptic_txs_features")
labels = spark.table("silver.elliptic_txs_classes")

# Keep only the label columns we need (this avoids duplicate column names)
labels = labels.select("txId", "class", "label_name")

# Join: attach each transaction's label to its features
joined = features.join(labels, on="txId", how="inner")

# Save as a silver table and count the rows
joined.write.mode("overwrite").saveAsTable("silver.transactions")
print(joined.count())

%md
## Step 6: Check the label counts
We count how many transactions have each label.
These should match the bronze profile: illicit 4545, licit 42019, unknown 157205.

In [0]:
%sql
SELECT label_name, COUNT(*) AS transactions
FROM silver.transactions
GROUP BY label_name

%md
## Step 7: More quality checks
We check the time steps, the label names, duplicate IDs, and missing values.
Expected result: 0 in every column.


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM silver.transactions WHERE time_step < 1 OR time_step > 49) AS bad_time_steps,
  (SELECT COUNT(*) FROM silver.transactions WHERE label_name NOT IN ('illicit', 'licit', 'unknown')) AS bad_labels,
  (SELECT COUNT(*) - COUNT(DISTINCT txId) FROM silver.transactions) AS duplicate_ids,
  (SELECT COUNT(*) FROM silver.transactions WHERE local_01 IS NULL OR agg_72 IS NULL) AS missing_features

## Step 7: More quality checks
Each cell checks one thing. The first three should show 0. The last one should show the same number twice.d


In [0]:
%sql
SELECT COUNT(*) AS bad_time_steps
FROM silver.transactions
WHERE time_step < 1 OR time_step > 49

In [0]:
%sql
SELECT COUNT(*) AS bad_labels
FROM silver.transactions
WHERE label_name NOT IN ('illicit', 'licit', 'unknown')

In [0]:
%sql
SELECT COUNT(*) AS missing_features
FROM silver.transactions
WHERE local_01 IS NULL OR agg_72 IS NULL

In [0]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT txId) AS unique_ids
FROM silver.transactions